<a target="_blank" href="https://colab.research.google.com/github/dcintlab/HS-25-26/blob/master/GoodFeaturesToTrack/edge_detected_video_with_mask.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>

# Corner Detection with Custom Edge Mask, PyTorch & Progress Tracking

This notebook demonstrates an advanced workflow for detecting corners in a video. It uses a pre-processed edge-detected video as a spatial mask to restrict the search area, improving accuracy and performance.

### Key Features:
- **PyTorch Integration:** Checks for GPU acceleration.
- **Robust Resource Management:** Uses `try...finally` blocks to ensure video files are closed properly.
- **Progress Feedback:** Implements `tqdm` for real-time visual progress monitoring.
- **VS Code Optimized:** Handles progress bar compatibility and environment-safe error management.

## Environment Setup

Setting up Google Drive mounting and automatic reloading of external modules.

In [1]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive/')
    %cd /content/drive/My Drive/HS-25-26
except:
    IN_COLAB = False

print(f'Running on {"Google colab" if IN_COLAB else "Local computer (VS Code)"}')

# Safe loading of autoreload extension
%load_ext autoreload
%autoreload 2

Running on Local computer (VS Code)


Failed to read module file 'c:\Users\HajbelBence\AppData\Local\Programs\Python\Python313\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\HajbelBence\AppData\Roaming\Python\Python313\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "C:\Users\HajbelBence\AppData\Roaming\Python\Python313\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "c:\Users\HajbelBence\AppData\Local\Programs\Python\Python313\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importli

## Libraries & Hardware Check

Importing core libraries and checking for CUDA (GPU) availability via PyTorch.

In [2]:
import cv2
import numpy as np
import torch
import sys

# Robust tqdm import for VS Code and standard Notebooks
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# Check for GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

Using device: cpu


## Configuration

Define your input file paths and the algorithm parameters.

In [3]:
# File paths
video_path = "chair2.mp4"
edge_video_path = "chair2_gpu_test_method.mp4"
output_path = "chair2_edge_detection_with_mask_last_version.mp4"

# Corner detection settings
max_corners = 10
quality_level = 0.1
min_distance = 20

# Morphological kernel for mask dilation
kernel = np.ones((3, 3), np.uint8)

## Video Stream Initialization

In [4]:
cap = cv2.VideoCapture(video_path)
edge_cap = cv2.VideoCapture(edge_video_path)

if not cap.isOpened() or not edge_cap.isOpened():
    cap.release()
    edge_cap.release()
    raise RuntimeError("Could not open input video files. Ensure files exist at the specified paths.")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) != 0 else 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

## Main Processing Pipeline

This section performs the frame-by-frame analysis and applies the corner detection algorithm using the custom mask.

In [5]:
frame_count = 0

try:
    # tqdm visual progress bar initialization
    with tqdm(total=total_frames, desc="Processing Video", unit="frame") as pbar:
        while True:
            ret1, frame = cap.read()
            ret2, edge_frame = edge_cap.read()

            if not ret1 or not ret2:
                break

            # ---- PRE-PROCESSING ----
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edge_gray = cv2.cvtColor(edge_frame, cv2.COLOR_BGR2GRAY)

            # Mask creation
            _, mask = cv2.threshold(edge_gray, 30, 255, cv2.THRESH_BINARY)
            mask = cv2.dilate(mask, kernel, iterations=1)

            # ---- FEATURE DETECTION ----
            corners = cv2.goodFeaturesToTrack(
                image=gray,
                maxCorners=max_corners,
                qualityLevel=quality_level,
                minDistance=min_distance,
                mask=mask
            )

            # ---- VISUALIZATION ----
            if corners is not None:
                corners = np.int32(corners)
                for x, y in corners.reshape(-1, 2):
                    cv2.circle(frame, (x, y), 7, (0, 0, 255), -1)

            out.write(frame)
            
            # Progress Bar update
            pbar.update(1)
            frame_count += 1

    print(f"\nSuccessfully processed {frame_count} frames.")

finally:
    # Resource release
    cap.release()
    edge_cap.release()
    out.release()
    print("Resource cleanup: Video streams released.")

Processing Video:   0%|          | 0/148 [00:00<?, ?frame/s]


Successfully processed 148 frames.
Resource cleanup: Video streams released.
